# Colab 15 — Natural-pair eval (AA + SS) on real CATH pairs

## What's new vs colab14

Colab14 evaluated SS in cross-representation using **perturbation pairs** —
one side a real SS sequence, the other side a synthetic perturbation of it.
That conflated the cross-rep signal with whatever generative regularities
the perturbation procedure imposed.

Colab15 evaluates on **natural biological pairs only**, using the
precomputed `ss_score` column in the CATH pair files (first iteration to do
so). The training pipeline is identical to colab14 (artificial-only,
band-weighted MSE, 30 epochs).

## Thesis question being tested

Training is on artificial AA-character pairs. If the SNN learned to
*approximate* normLev abstractly (rather than memorize AA-alphabet
regularities), eval on the SS representation of real CATH pairs should also
work despite the 3-letter alphabet shift. The AA-vs-SS comparison on the
same biological pairs is the central cross-representation test.

## Eval composition (from `sampledata/cath/cath_eval.csv.gz`)

- **high** (aa_score ≥ 0.70): 5 pairs (4 strictly-filtered + 1 short-length-rescue)
- **mid** [0.30, 0.70): 2,945 pairs (all available)
- **far** (<0.30): 2,000 pairs (capped, seed=42 sample from 23K candidates)

Bins defined on `aa_score`. Same pairs are evaluated in both representations.

## k-NN retrieval

For each of the 5 high-AA pairs (10 queries with both directions),
retrieve top-k partners from the full ~10K protein pool. Run separately on
the AA and SS representations. Random baseline at top-50 ≈ 0.5%.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
          'cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from rapidfuzz.distance import Levenshtein as RFLevenshtein

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants — including band thresholds and weights

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
N_HELD = 5000
NUM_EPOCHS = 30
BATCH_SIZE = 128

# Band thresholds + weights for the new banded MSE loss
BAND_LOW  = 0.30    # below this = far (don't care)
BAND_HIGH = 0.70    # at/above this = high (care most)
W_FAR  = 0.5
W_MID  = 2.0
W_HIGH = 4.0

# Top-k retrieval setup
TOPK_N_QUERIES   = 50
TOPK_N_CANDIDATES = 500
TOPK_VALUES = (1, 5, 10)

print(f'Bands: far<{BAND_LOW}, mid in [{BAND_LOW},{BAND_HIGH}), high>={BAND_HIGH}')
print(f'Weights: far={W_FAR}, mid={W_MID}, high={W_HIGH}')

## 3. Helpers — Levenshtein, perturb, encode, band labeling

In [ ]:
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - levenshtein(a, b) / L

def fast_norm_lev(a, b):
    """rapidfuzz-backed normLev (C implementation, ~100x faster)."""
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in abc if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

def random_aa_seq(min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

def band_label(x):
    """Return 'far', 'mid', or 'high' for a similarity value (numpy/python scalar)."""
    if x < BAND_LOW:  return 'far'
    if x < BAND_HIGH: return 'mid'
    return 'high'

def band_labels_arr(xs):
    xs = np.asarray(xs)
    out = np.full(xs.shape, 'mid', dtype=object)
    out[xs < BAND_LOW]  = 'far'
    out[xs >= BAND_HIGH] = 'high'
    return out

## 4. Build training pairs — single-procedure, target-uniform sampling

Unchanged from colab13. Sample `t ∼ Uniform[0, 1]`, pick `k = round((1−t)·L)`, apply `perturb`, and use the actual computed `normLev` as the label. All training pairs come from the same procedure (no procedural axis the encoder can shortcut on).

The achievable label range is `[~0.28, 1.0]` because of the alphabet-entropy floor. That is intentional: below 0.28 is the "far" band where we explicitly don't care about resolution — the band-weighted loss will only need to teach the model "this is far" rather than "this is exactly 0.13".

In [ ]:
def make_targeted_perturbation_pairs(seed_fn, alphabet, n_pairs):
    pairs = []
    attempts = 0
    max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = seed_fn()
        if seed is None or len(seed) < MIN_LEN: continue
        L = len(seed)
        t = float(rng.uniform(0.0, 1.0))
        k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet)
        if 1 <= len(other) <= MAX_LEN:
            label = norm_lev(seed, other)
            pairs.append((seed, other, label))
    return pairs

print('Building training pairs (target-uniform random-AA perturbations)…')
train_pairs = make_targeted_perturbation_pairs(
    seed_fn=random_aa_seq, alphabet=AA_ALPHABET, n_pairs=N_TRAIN)
print(f'  got {len(train_pairs)} training pairs')

labels = np.array([p[2] for p in train_pairs])
band_counts = {b: int((band_labels_arr(labels) == b).sum()) for b in ('far','mid','high')}
print(f'  label range:    [{labels.min():.3f}, {labels.max():.3f}]')
print(f'  band counts:    {band_counts}')

plt.figure(figsize=(8, 3))
plt.hist(labels, bins=40, edgecolor='k', alpha=0.75)
plt.axvline(BAND_LOW,  color='r', ls='--', label=f'far/mid threshold ({BAND_LOW})')
plt.axvline(BAND_HIGH, color='g', ls='--', label=f'mid/high threshold ({BAND_HIGH})')
plt.xlabel('Label (normLev similarity)'); plt.ylabel('Count')
plt.title('Training-pair label distribution with band thresholds')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Dataset / Architecture (unchanged from colab13)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        a, b, label = self.pairs[i]
        return encode_pad(a), encode_pad(b), torch.tensor(label, dtype=torch.float32)

train_dl = DataLoader(PairDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True)

class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 max_len=MAX_LEN, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2 * max_len, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        x = self.embedding(x).permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x * mask.unsqueeze(1)
        x = x.flatten(1)
        x = self.fc(x)
        return F.normalize(x, p=2, dim=1)

class SiameseNetwork(nn.Module):
    def __init__(self, **kw):
        super().__init__(); self.encoder = SiameseEncoder(**kw)
    def forward(self, a, b):
        ea, eb = self.encoder(a), self.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0
    def encode(self, x): return self.encoder(x)

model = SiameseNetwork().to(device)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Training — band-weighted MSE

For each pair, the squared error term is multiplied by a per-sample weight determined by the *true* label's band:

- label `< 0.30`: weight `0.5` (far — we don't care about resolution)
- `0.30 ≤ label < 0.70`: weight `2.0` (mid — twilight zone)
- label `≥ 0.70`: weight `4.0` (high — biologically meaningful)

The model is forced to spend its capacity where it matters most. Plain MSE is recovered by setting all weights to 1.0.

In [ ]:
def banded_mse_loss(pred, label):
    weights = torch.ones_like(label)
    weights[label < BAND_LOW] = W_FAR
    weights[(label >= BAND_LOW) & (label < BAND_HIGH)] = W_MID
    weights[label >= BAND_HIGH] = W_HIGH
    return (((pred - label) ** 2) * weights).mean()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
model.train()
for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0; nb = 0
    for a, b, label in train_dl:
        a = a.to(device); b = b.to(device); label = label.to(device)
        pred = model(a, b)
        loss = banded_mse_loss(pred, label)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item(); nb += 1
    avg = epoch_loss / nb
    losses.append(avg)
    if epoch % 2 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} — banded-MSE: {avg:.5f}')

plt.figure(figsize=(8, 4))
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('Band-weighted MSE')
plt.title('Training loss — band-weighted MSE')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final loss: {losses[-1]:.5f}')

## 7. Load eval set + protein pool

Loads the committed eval set (`cath_eval.csv.gz`) and builds the
combined train70+test30 protein pool. Training never saw any CATH protein,
so the train/test split has no leakage role and is dropped.

The protein pool for k-NN retrieval includes the two short-length proteins
from the rescued high-AA pair (below MIN_LEN=50). They're flagged as such.

In [ ]:
import os, pandas as pd

# Build the combined protein pool
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
prot_df = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')

# Standard-char + length-match filter (same as colab14)
prot_df = prot_df[prot_df['aa_seq'].apply(is_standard_aa)]
prot_df = prot_df[prot_df['ss_seq'].apply(is_standard_ss)]
prot_df = prot_df[prot_df['aa_seq'].str.len() == prot_df['ss_seq'].str.len()]
prot_df['aa_len'] = prot_df['aa_seq'].str.len()

# Eligible = in [MIN_LEN, MAX_LEN] OR explicitly whitelisted as rescued
RESCUED = {'4z0mC02', '3qkaE02'}   # from build_eval_set.py
in_range = prot_df['aa_len'].between(MIN_LEN, MAX_LEN)
prot_df = prot_df[in_range | prot_df['domain_id'].isin(RESCUED)].reset_index(drop=True)

id_to_aa = dict(zip(prot_df['domain_id'], prot_df['aa_seq']))
id_to_ss = dict(zip(prot_df['domain_id'], prot_df['ss_seq']))
all_domains = list(prot_df['domain_id'])
print(f'Protein pool: {len(prot_df)} (incl. {len(prot_df) - in_range.sum()} rescued short)')

# Load the eval set
eval_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
print(f"Eval pairs: {len(eval_df)}")
print(eval_df['aa_bin'].value_counts().reindex(['high', 'mid', 'far']).to_string())

## 8. Predict helper + banded metrics

Same `predict_pairs` / `banded_metrics` as colab14, with one addition:
per-bin **prediction-bias** diagnostic (`mean(pred) − mean(true)` + `std(pred)`
per band). Bias is load-bearing for the benchmarks table — printed plus
plotted later.

In [ ]:
def predict_pairs(pairs, batch_size=512):
    """Each pair is (seq_a, seq_b, true_label)."""
    model.eval()
    true_v, pred_v = [], []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
            b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
            pred = model(a, b).cpu().numpy()
            true_v.extend([p[2] for p in batch])
            pred_v.extend(pred.tolist())
    return np.array(true_v), np.array(pred_v)

def per_band_bias(true_v, pred_v):
    """Return dict of (mean(pred)-mean(true), pred.std()) per band, on TRUE bin."""
    tb = band_labels_arr(true_v)
    out = {}
    for b in ('far', 'mid', 'high'):
        m = (tb == b)
        if m.sum() == 0:
            out[b] = {'n': 0, 'true_mean': float('nan'), 'pred_mean': float('nan'),
                      'bias': float('nan'), 'pred_std': float('nan')}
            continue
        out[b] = {
            'n': int(m.sum()),
            'true_mean': float(true_v[m].mean()),
            'pred_mean': float(pred_v[m].mean()),
            'bias':      float(pred_v[m].mean() - true_v[m].mean()),
            'pred_std':  float(pred_v[m].std()),
        }
    return out

def banded_metrics(name, true_v, pred_v):
    if len(true_v) < 10:
        print(f'{name}: n={len(true_v)} too few'); return None
    pr_all, _ = pearsonr(true_v, pred_v)
    sr_all, _ = spearmanr(true_v, pred_v)
    hi = true_v >= BAND_HIGH
    mi = (true_v >= BAND_LOW) & (true_v < BAND_HIGH)
    fa = true_v < BAND_LOW
    pr_hi = pearsonr(true_v[hi], pred_v[hi])[0] if hi.sum() > 10 else float('nan')
    pr_mi = pearsonr(true_v[mi], pred_v[mi])[0] if mi.sum() > 10 else float('nan')
    pr_fa = pearsonr(true_v[fa], pred_v[fa])[0] if fa.sum() > 10 else float('nan')
    mae_hi = np.abs(pred_v[hi] - true_v[hi]).mean() if hi.sum() > 0 else float('nan')
    mae_mi = np.abs(pred_v[mi] - true_v[mi]).mean() if mi.sum() > 0 else float('nan')
    mae_fa = np.abs(pred_v[fa] - true_v[fa]).mean() if fa.sum() > 0 else float('nan')
    y_hi = (true_v >= BAND_HIGH).astype(int)
    auroc = roc_auc_score(y_hi, pred_v) if 0 < y_hi.sum() < len(y_hi) else float('nan')
    bias = per_band_bias(true_v, pred_v)

    print(f'--- {name} (n={len(true_v)}) ---')
    print(f'  Pearson r (all):  {pr_all:+.3f}    Spearman: {sr_all:+.3f}')
    print(f'  Per-bin r:  high={pr_hi:+.3f}  mid={pr_mi:+.3f}  far={pr_fa:+.3f}')
    print(f'  Per-bin MAE: high={mae_hi:.3f}  mid={mae_mi:.3f}  far={mae_fa:.3f}')
    print(f'  AUROC (is-high): {auroc:.3f}')
    print(f'  Prediction-bias per band (mean_pred - mean_true | pred_std):')
    for b in ('high', 'mid', 'far'):
        v = bias[b]
        print(f'    {b:>4}: n={v["n"]:>5}  bias={v["bias"]:+.3f}  pred_std={v["pred_std"]:.3f}')
    return {
        'pr_all': pr_all, 'sr_all': sr_all,
        'pr_hi': pr_hi, 'pr_mi': pr_mi, 'pr_fa': pr_fa,
        'mae_hi': mae_hi, 'mae_mi': mae_mi, 'mae_fa': mae_fa,
        'auroc': auroc, 'bias': bias, 'n': len(true_v),
    }

## 9. Build (seq_a, seq_b, label) pair lists from eval_df

Two pair lists from the same `eval_df`: one with AA sequences and `aa_score`
labels, one with SS sequences and `ss_score` labels.

In [ ]:
def make_pairs(df, rep):
    """rep ∈ {'aa', 'ss'} selects the sequence side + the score column."""
    score_col = 'aa_score' if rep == 'aa' else 'ss_score'
    lookup = id_to_aa if rep == 'aa' else id_to_ss
    out = []
    for _, r in df.iterrows():
        a_id, b_id = r['domain_a'], r['domain_b']
        if a_id not in lookup or b_id not in lookup: continue
        out.append((lookup[a_id], lookup[b_id], float(r[score_col])))
    return out

eval_pairs_aa = make_pairs(eval_df, 'aa')
eval_pairs_ss = make_pairs(eval_df, 'ss')
print(f'AA pairs: {len(eval_pairs_aa)}    SS pairs: {len(eval_pairs_ss)}')

## 10. AA-side eval — predicted vs aa_score

In-representation test. Training was on AA characters, so this is the
"fair" eval. Expectation: mid band well-resolved, high band perfect on the
5 available pairs, far band predicted-low (high variance acceptable).

In [ ]:
true_aa, pred_aa = predict_pairs(eval_pairs_aa)
m_aa = banded_metrics('AA representation (label=aa_score)', true_aa, pred_aa)

fig, ax = plt.subplots(figsize=(6, 6))
colors_bin = {'far': 'tab:gray', 'mid': 'tab:orange', 'high': 'tab:red'}
tb_aa = band_labels_arr(true_aa)
for b in ('far', 'mid', 'high'):
    m = tb_aa == b
    ax.scatter(true_aa[m], pred_aa[m], s=12 if b != 'high' else 80,
               alpha=0.35 if b != 'high' else 1.0, color=colors_bin[b],
               edgecolors='black' if b == 'high' else 'none', linewidths=0.6,
               label=f'{b} (n={m.sum()})')
ax.plot([0, 1], [0, 1], 'r-', lw=1.5, label='y = x')
ax.axhline(BAND_LOW,  color='k', ls=':', lw=0.5)
ax.axhline(BAND_HIGH, color='k', ls=':', lw=0.5)
ax.axvline(BAND_LOW,  color='k', ls=':', lw=0.5)
ax.axvline(BAND_HIGH, color='k', ls=':', lw=0.5)
ax.set_xlabel('True aa_score (normLev)')
ax.set_ylabel('Predicted similarity')
ax.set_title(f'AA eval — Pearson r = {m_aa["pr_all"]:+.3f}')
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 11. SS-side eval — predicted vs ss_score

Cross-representation test. Same biological pairs as section 10, but
sequences are SS strings (3-letter alphabet H/L/S), labels are precomputed
`ss_score` from the pair files. This is the **central thesis test**.

If SS-side high-band MAE is comparable to AA-side high-band MAE, the
SNN learned to apply Levenshtein abstractly — not just memorize
AA-alphabet structure.

In [ ]:
true_ss, pred_ss = predict_pairs(eval_pairs_ss)
m_ss = banded_metrics('SS representation (label=ss_score)', true_ss, pred_ss)

fig, ax = plt.subplots(figsize=(6, 6))
tb_ss = band_labels_arr(true_ss)
for b in ('far', 'mid', 'high'):
    m = tb_ss == b
    ax.scatter(true_ss[m], pred_ss[m], s=12 if b != 'high' else 80,
               alpha=0.35 if b != 'high' else 1.0, color=colors_bin[b],
               edgecolors='black' if b == 'high' else 'none', linewidths=0.6,
               label=f'{b} (n={m.sum()})')
ax.plot([0, 1], [0, 1], 'r-', lw=1.5, label='y = x')
ax.axhline(BAND_LOW,  color='k', ls=':', lw=0.5)
ax.axhline(BAND_HIGH, color='k', ls=':', lw=0.5)
ax.axvline(BAND_LOW,  color='k', ls=':', lw=0.5)
ax.axvline(BAND_HIGH, color='k', ls=':', lw=0.5)
ax.set_xlabel('True ss_score (normLev)')
ax.set_ylabel('Predicted similarity')
ax.set_title(f'SS eval — Pearson r = {m_ss["pr_all"]:+.3f}')
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 12. Prediction-bias diagnostic — figure

Per-bin bias (`mean(pred) - mean(true)`) for AA and SS side-by-side.
Sign tells us whether the model systematically over- or under-predicts
each band. Magnitude tells us how compressed predictions are.

In [ ]:
bands = ['high', 'mid', 'far']
aa_bias = [m_aa['bias'][b]['bias'] for b in bands]
ss_bias = [m_ss['bias'][b]['bias'] for b in bands]
aa_std  = [m_aa['bias'][b]['pred_std'] for b in bands]
ss_std  = [m_ss['bias'][b]['pred_std'] for b in bands]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(bands)); w = 0.35
axL.bar(x - w/2, aa_bias, w, label='AA',  color='tab:blue')
axL.bar(x + w/2, ss_bias, w, label='SS',  color='tab:green')
axL.axhline(0, color='k', lw=0.5)
axL.set_xticks(x); axL.set_xticklabels(bands)
axL.set_ylabel('mean(pred) − mean(true)')
axL.set_title('Per-bin prediction bias')
axL.legend(); axL.grid(True, alpha=0.3)

axR.bar(x - w/2, aa_std, w, label='AA', color='tab:blue')
axR.bar(x + w/2, ss_std, w, label='SS', color='tab:green')
axR.set_xticks(x); axR.set_xticklabels(bands)
axR.set_ylabel('std(pred)')
axR.set_title('Per-bin prediction spread')
axR.legend(); axR.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('\nBenchmarks table (paste into BENCHMARKS.md):')
print(f"{'bin':<5} {'n':>5} {'AA_bias':>9} {'AA_std':>8} {'SS_bias':>9} {'SS_std':>8}")
for b in bands:
    print(f"{b:<5} {m_aa['bias'][b]['n']:>5} "
          f"{m_aa['bias'][b]['bias']:>+9.3f} {m_aa['bias'][b]['pred_std']:>8.3f} "
          f"{m_ss['bias'][b]['bias']:>+9.3f} {m_ss['bias'][b]['pred_std']:>8.3f}")

## 13. Identity pair test — sanity check

Each protein paired with itself should yield similarity = 1.0. One-line
sanity check; if this fails the model is broken.

In [ ]:
from random import sample as rsample
sample_ids = rsample(all_domains, 100)
id_pairs_aa = [(id_to_aa[d], id_to_aa[d], 1.0) for d in sample_ids]
id_pairs_ss = [(id_to_ss[d], id_to_ss[d], 1.0) for d in sample_ids]
_, p_aa = predict_pairs(id_pairs_aa); _, p_ss = predict_pairs(id_pairs_ss)
print(f'Identity test AA: mean={p_aa.mean():.4f} ± {p_aa.std():.4f}')
print(f'Identity test SS: mean={p_ss.mean():.4f} ± {p_ss.std():.4f}')

## 14. k-NN retrieval — AA representation

For each protein in the 5 high-AA pairs, encode it (via AA sequence) and
rank all ~10K candidates by predicted similarity. Check the rank of its
true partner.

10 queries total (5 pairs × 2 directions). Random baseline:
top-1 ≈ 0.01%, top-10 ≈ 0.1%, top-50 ≈ 0.5%.

In [ ]:
HIGH_PAIRS = eval_df[eval_df['aa_bin'] == 'high'][['domain_a', 'domain_b']].values.tolist()
print(f'High-AA pairs for retrieval ({len(HIGH_PAIRS)}):')
for a, b in HIGH_PAIRS: print(f'  {a} ↔ {b}')

def encode_pool(seq_lookup, ids, batch_size=256):
    """Encode every protein in `ids` via the provided sequence lookup. Returns (N, d) tensor on device."""
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ids), batch_size):
            chunk = ids[i:i+batch_size]
            x = torch.stack([encode_pad(seq_lookup[d]) for d in chunk]).to(device)
            out.append(model.encode(x))
    return torch.cat(out, 0)

def predicted_sim_to_all(query_emb, pool_emb):
    """For one query embedding (d,) and pool (N, d), return (N,) predicted sims."""
    diff = pool_emb - query_emb.unsqueeze(0)
    return 1.0 - torch.linalg.vector_norm(diff, ord=2, dim=1) / 2.0

def run_retrieval(seq_lookup, label):
    pool_emb = encode_pool(seq_lookup, all_domains)
    id_to_idx = {d: i for i, d in enumerate(all_domains)}
    K_VALUES = (1, 5, 10, 50)
    hits = {k: 0 for k in K_VALUES}
    ranks = []
    print(f'\n--- {label} retrieval (pool={len(all_domains)}) ---')
    for a, b in HIGH_PAIRS:
        for q, partner in [(a, b), (b, a)]:
            if q not in id_to_idx or partner not in id_to_idx:
                print(f'  SKIP {q}→{partner}: not in pool'); continue
            qi, pi = id_to_idx[q], id_to_idx[partner]
            sims = predicted_sim_to_all(pool_emb[qi], pool_emb).cpu().numpy()
            sims[qi] = -np.inf
            order = np.argsort(-sims)
            rank = int(np.where(order == pi)[0][0]) + 1
            ranks.append((q, partner, rank))
            for k in K_VALUES:
                if rank <= k: hits[k] += 1
            print(f'  {q} → {partner}:  rank={rank}  predicted_sim={sims[pi]:.3f}')
    n_q = len(ranks)
    print(f'\n  Summary over {n_q} queries:')
    for k in K_VALUES:
        baseline = k / len(all_domains)
        print(f'    hits@{k:>2}: {hits[k]}/{n_q}  ({hits[k]/n_q:.1%})    random baseline {baseline:.4%}')
    return ranks, hits

ranks_aa, hits_aa = run_retrieval(id_to_aa, 'AA representation')

## 15. k-NN retrieval — SS representation (cross-rep)

Same 10 queries, same partners, same pool — but encoded via SS sequences
instead of AA. The 3-letter alphabet (H/L/S) gets mapped to AA indices the
same way colab14 did it. If SS retrieval *also* hits the partners in
top-k, the encoder learned something abstract about Levenshtein, not just
AA-character co-occurrence.

In [ ]:
ranks_ss, hits_ss = run_retrieval(id_to_ss, 'SS representation')

## 16. PCA — pair-difference vectors (AA)

Sanity check: project (ea - eb) into 2D for the AA eval pairs and color by
true band. If the band ordering is visible along PC1, the function is
represented in the embedding geometry.

In [ ]:
def get_pair_embeddings(pairs, batch_size=256):
    model.eval()
    eas, ebs = [], []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
            b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
            eas.append(model.encode(a).cpu().numpy())
            ebs.append(model.encode(b).cpu().numpy())
    return np.concatenate(eas, 0), np.concatenate(ebs, 0)

ea, eb = get_pair_embeddings(eval_pairs_aa)
diff = ea - eb
pca = PCA(n_components=2)
proj = pca.fit_transform(diff)

fig, ax = plt.subplots(figsize=(7, 6))
tb = band_labels_arr(true_aa)
for b in ('far', 'mid', 'high'):
    m = tb == b
    ax.scatter(proj[m, 0], proj[m, 1], s=12 if b != 'high' else 80,
               alpha=0.35 if b != 'high' else 1.0, color=colors_bin[b],
               edgecolors='black' if b == 'high' else 'none', linewidths=0.6,
               label=f'{b} (n={m.sum()})')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA of pair-difference vectors (AA eval)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 17. Summary table

Side-by-side comparison of AA vs SS eval for the BENCHMARKS.md table.

In [ ]:
rows = [
    ('Pearson r (all)',  f"{m_aa['pr_all']:+.3f}",  f"{m_ss['pr_all']:+.3f}"),
    ('Spearman ρ',       f"{m_aa['sr_all']:+.3f}",  f"{m_ss['sr_all']:+.3f}"),
    ('Per-bin r (high)', f"{m_aa['pr_hi']:+.3f}",   f"{m_ss['pr_hi']:+.3f}"),
    ('Per-bin r (mid)',  f"{m_aa['pr_mi']:+.3f}",   f"{m_ss['pr_mi']:+.3f}"),
    ('Per-bin r (far)',  f"{m_aa['pr_fa']:+.3f}",   f"{m_ss['pr_fa']:+.3f}"),
    ('MAE (high)',       f"{m_aa['mae_hi']:.3f}",   f"{m_ss['mae_hi']:.3f}"),
    ('MAE (mid)',        f"{m_aa['mae_mi']:.3f}",   f"{m_ss['mae_mi']:.3f}"),
    ('MAE (far)',        f"{m_aa['mae_fa']:.3f}",   f"{m_ss['mae_fa']:.3f}"),
    ('AUROC is-high',    f"{m_aa['auroc']:.3f}",    f"{m_ss['auroc']:.3f}"),
    ('Bias (high)',      f"{m_aa['bias']['high']['bias']:+.3f}",
                         f"{m_ss['bias']['high']['bias']:+.3f}"),
    ('Bias (mid)',       f"{m_aa['bias']['mid']['bias']:+.3f}",
                         f"{m_ss['bias']['mid']['bias']:+.3f}"),
    ('Bias (far)',       f"{m_aa['bias']['far']['bias']:+.3f}",
                         f"{m_ss['bias']['far']['bias']:+.3f}"),
    ('Retrieval hits@10', f"{hits_aa[10]}/{len(ranks_aa)}",
                          f"{hits_ss[10]}/{len(ranks_ss)}"),
    ('Retrieval hits@50', f"{hits_aa[50]}/{len(ranks_aa)}",
                          f"{hits_ss[50]}/{len(ranks_ss)}"),
]
print(f"{'metric':<22} {'AA':>10} {'SS':>10}")
print('-' * 44)
for r in rows: print(f"{r[0]:<22} {r[1]:>10} {r[2]:>10}")